# PSM Memory LOCOMO benchmark on Colab

This notebook clones the PSM repo, builds the local workspace packages, installs those local package tarballs, restores an already-ingested LOCOMO SQLite DB from Hugging Face Hub, and runs retrieval plus answer-generation benchmark evaluation.

Recommended Colab runtime: CPU is enough for benchmarking an existing DB. GPU is only needed if you re-run ingestion.

In [ ]:
# Install Node.js 22. Colab's default Node can be older than the package engine range.
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node --version
!npm --version

In [ ]:
# Keep caches under /content so the paths are predictable.
# Hugging Face Hub is the portable artifact store across Colab accounts.
# Build PSM from the cloned repo and install local package tarballs instead of npm latest.
HF_REPO_ID = "chkrishna2001/psm-memory-locomo-artifacts"  # Change to your private HF dataset repo if needed.
HF_LOCAL_DIR = "/content/locomo/hf-artifacts"
WORK_DIR = "/content/psm-memory-work"
REPO_DIR = "/content/PSM"
%env PSM_MEMORY_HOME=/content/psm-memory-cache
!mkdir -p "$WORK_DIR" /content/locomo/results "$HF_LOCAL_DIR"
!rm -rf "$REPO_DIR"
!git clone --depth 1 https://github.com/chkrishna2001/PSM.git "$REPO_DIR"
%cd /content/PSM
!npm ci
!npm run build
!rm -f "$WORK_DIR"/psm-memory-*.tgz
!npm pack -w @psm-memory/sdk --pack-destination "$WORK_DIR"
!npm pack -w @psm-memory/cli --pack-destination "$WORK_DIR"
%cd /content/psm-memory-work
!npm init -y >/dev/null 2>&1 || true
!npm install ./psm-memory-sdk-*.tgz ./psm-memory-cli-*.tgz


In [ ]:
# Configure Hugging Face Hub artifact sync. This lets a different Colab Google account resume the same run.
!pip -q install -U huggingface_hub
import getpass
import os
import shutil
import subprocess
from pathlib import Path

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN') or (userdata.get('HF_TOKEN') or userdata.get('HUGGINGFACE_TOKEN') or '')
except Exception:
    pass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')

os.environ['HF_REPO_ID'] = HF_REPO_ID
os.environ['HF_LOCAL_DIR'] = HF_LOCAL_DIR
!huggingface-cli repo create "$HF_REPO_ID" --type dataset --private -y >/dev/null 2>&1 || true

def hf_download(path_in_repo, local_dir):
    cmd = [
        'huggingface-cli', 'download', HF_REPO_ID,
        '--repo-type', 'dataset',
        '--local-dir', local_dir,
        path_in_repo,
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode == 0:
        print(f"Restored hf://{HF_REPO_ID}/{path_in_repo}")
        return True
    print(f"No HF artifact yet: {path_in_repo}")
    return False

Path(HF_LOCAL_DIR).mkdir(parents=True, exist_ok=True)
hf_download('data/locomo-psm-memory-ingest.db', HF_LOCAL_DIR)
hf_download('ingest/checkpoints/locomo-psm-memory-ingest.db', HF_LOCAL_DIR)
hf_download('benchmark/results/locomo-answer-results-smoke-20-json-v2.json', HF_LOCAL_DIR)
hf_download('benchmark/results/locomo-results-smoke-20-json-v2.json', HF_LOCAL_DIR)
hf_download('benchmark/results/locomo-comparison-smoke-20-json-v2.md', HF_LOCAL_DIR)
Path('/content/locomo/results').mkdir(parents=True, exist_ok=True)

for candidate in [
    Path(HF_LOCAL_DIR) / 'data' / 'locomo-psm-memory-ingest.db',
    Path(HF_LOCAL_DIR) / 'ingest' / 'checkpoints' / 'locomo-psm-memory-ingest.db',
]:
    if candidate.exists():
        shutil.copy2(candidate, '/content/locomo/results/locomo-psm-memory-ingest.db')
        print(f"Restored benchmark DB from HF: {candidate}")
        break

hf_results = Path(HF_LOCAL_DIR) / 'benchmark' / 'results'
if hf_results.exists():
    for file in hf_results.iterdir():
        if file.is_file():
            shutil.copy2(file, Path('/content/locomo/results') / file.name)
    print(f"Restored benchmark result files from HF: {hf_results}")

sync_script = Path('/content/hf_sync_benchmark.sh')
sync_script.write_text(f'''#!/usr/bin/env bash
set +e
while true; do
  mkdir -p "{HF_LOCAL_DIR}/benchmark/results" "{HF_LOCAL_DIR}/data"
  cp -f /content/locomo/results/locomo-psm-memory-ingest.db "{HF_LOCAL_DIR}/data/locomo-psm-memory-ingest.db" 2>/dev/null || true
  cp -f /content/locomo/results/* "{HF_LOCAL_DIR}/benchmark/results/" 2>/dev/null || true
  huggingface-cli upload "{HF_REPO_ID}" "{HF_LOCAL_DIR}/benchmark" benchmark --repo-type dataset --commit-message "sync benchmark artifacts" >/tmp/hf-sync-benchmark.log 2>&1 || true
  huggingface-cli upload "{HF_REPO_ID}" "{HF_LOCAL_DIR}/data" data --repo-type dataset --commit-message "sync benchmark db" >/tmp/hf-sync-data.log 2>&1 || true
  sleep 60
done
''')
sync_script.chmod(0o755)
!pkill -f hf_sync_benchmark.sh >/dev/null 2>&1 || true
!nohup bash /content/hf_sync_benchmark.sh >/tmp/hf-sync-benchmark.nohup 2>&1 &
print(f"HF background sync started for dataset repo: {HF_REPO_ID}")

In [ ]:
# Ensure the locally built CLI can prepare the embedding/model cache if benchmark helpers need it.
!./node_modules/.bin/psm-memory setup --pretty

In [ ]:
# Copy benchmark data and the Colab harness from the same cloned repo revision.
!cp /content/PSM/benchmark/locomo/colab/locomo-benchmark.mjs /content/psm-memory-work/locomo-benchmark.mjs
!ls -lh /content/PSM/benchmark/locomo/data/locomo10.json /content/psm-memory-work/locomo-benchmark.mjs
!node -e "console.log(require('fs').existsSync('/content/psm-memory-work/node_modules/@psm-memory/sdk/dist/index.js') ? 'local sdk installed' : 'missing sdk')"

In [ ]:
# Restore the already-ingested LOCOMO DB from Hugging Face. This skips ingestion.
from pathlib import Path
import shutil
import sqlite3

DB = "/content/locomo/results/locomo-psm-memory-ingest.db"
DATA = "/content/PSM/benchmark/locomo/data/locomo10.json"

Path(DB).parent.mkdir(parents=True, exist_ok=True)
if Path(DB).exists():
    print(f"Using DB already restored locally: {DB}")
else:
    raise FileNotFoundError(f"DB not found in HF restore: {DB}. Upload it to hf://{HF_REPO_ID}/data/locomo-psm-memory-ingest.db first.")
with sqlite3.connect(DB) as conn:
    tables = {row[0] for row in conn.execute("select name from sqlite_master where type='table'")}
    required = {"episodic", "semantic", "decisions", "memory_embeddings"}
    missing = sorted(required - tables)
    if missing:
        raise RuntimeError(f"Restored DB is not a PSM memory DB. Missing tables: {missing}. Found tables: {sorted(tables)}")
    episodic_count = conn.execute("select count(*) from episodic").fetchone()[0]
    embedding_count = conn.execute("select count(*) from memory_embeddings").fetchone()[0]
    print(f"episodic rows: {episodic_count:,}")
    print(f"embedding rows: {embedding_count:,}")
print(f"Local DB: {DB}")
print(f"DB size: {Path(DB).stat().st_size:,} bytes")

In [ ]:
# Evaluate evidence retrieval over the memory DB.
# Start with a cheap answer-evaluation smoke test before spending tokens on all 1,982 questions.
# Set ANSWER_LIMIT = 0 only after the smoke output looks good.
ANSWER_LIMIT = 20
ANSWER_TOP_K = 10
ANSWER_CONTEXT_K = 5
OPENROUTER_DELAY_MS = 4000
OPENROUTER_MAX_RETRIES = 8
CHECKPOINT_EVERY = 5
RUN_LABEL = "full-json-v2" if ANSWER_LIMIT == 0 else f"smoke-{ANSWER_LIMIT}-json-v2"

OUT = f"/content/locomo/results/locomo-results-{RUN_LABEL}.json"
ANSWER_OUT = f"/content/locomo/results/locomo-answer-results-{RUN_LABEL}.json"
REPORT = f"/content/locomo/results/locomo-comparison-{RUN_LABEL}.md"
BASELINES = "/content/PSM/benchmark/locomo/baselines/memory-tools.json"
ANSWER_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
JUDGE_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

import getpass
import os
for key, value in {
    "DATA": DATA,
    "DB": DB,
    "OUT": OUT,
    "ANSWER_OUT": ANSWER_OUT,
    "REPORT": REPORT,
    "BASELINES": BASELINES,
    "ANSWER_TOP_K": str(ANSWER_TOP_K),
    "ANSWER_CONTEXT_K": str(ANSWER_CONTEXT_K),
    "OPENROUTER_DELAY_MS": str(OPENROUTER_DELAY_MS),
    "OPENROUTER_MAX_RETRIES": str(OPENROUTER_MAX_RETRIES),
    "ANSWER_LIMIT": str(ANSWER_LIMIT),
    "CHECKPOINT_EVERY": str(CHECKPOINT_EVERY),
    "ANSWER_MODEL": ANSWER_MODEL,
    "JUDGE_MODEL": JUDGE_MODEL,
    "RUN_LABEL": RUN_LABEL,
}.items():
    os.environ[key] = value

print(f"Benchmark DB: {DB}")
!ls -lh "$DB"

try:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = os.environ.get('OPENROUTER_API_KEY') or (userdata.get('OPENROUTER_API_KEY') or '')
except Exception:
    pass
if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OPENROUTER_API_KEY: ')

# Resume from any result JSON restored from HF. The background HF sync uploads checkpoints while this runs.
!set -e; \
node /content/psm-memory-work/locomo-benchmark.mjs evaluate --data "$DATA" --db "$DB" --out "$OUT" --top-k 3; \
node /content/psm-memory-work/locomo-benchmark.mjs answer-evaluate --data "$DATA" --db "$DB" --out "$ANSWER_OUT" --top-k $ANSWER_TOP_K --answer-context-k $ANSWER_CONTEXT_K --request-delay-ms $OPENROUTER_DELAY_MS --request-max-retries $OPENROUTER_MAX_RETRIES --limit $ANSWER_LIMIT --answer-model "$ANSWER_MODEL" --judge-model "$JUDGE_MODEL" --checkpoint-every $CHECKPOINT_EVERY --resume false; \
node /content/psm-memory-work/locomo-benchmark.mjs report --psm "$ANSWER_OUT" --baselines "$BASELINES" --out "$REPORT"; \
mkdir -p "$HF_LOCAL_DIR/benchmark/results" "$HF_LOCAL_DIR/data"; \
cp "$DB" "$HF_LOCAL_DIR/data/locomo-psm-memory-ingest.db" 2>/dev/null || true; \
cp /content/locomo/results/* "$HF_LOCAL_DIR/benchmark/results/" 2>/dev/null || true; \
huggingface-cli upload "$HF_REPO_ID" "$HF_LOCAL_DIR/benchmark" benchmark --repo-type dataset --commit-message "final benchmark artifacts" || true; \
huggingface-cli upload "$HF_REPO_ID" "$HF_LOCAL_DIR/data" data --repo-type dataset --commit-message "final benchmark db" || true

In [ ]:
# Inspect output artifacts.
!ls -lh /content/locomo/results
!find "$HF_LOCAL_DIR" -maxdepth 3 -type f -printf '%p %k KB\n' 2>/dev/null | sort | tail -50
import json
from pathlib import Path
run_label = globals().get('RUN_LABEL', 'smoke-50-exact')
summary = Path('/content/locomo/results/ingest-summary.json')
results = Path(f'/content/locomo/results/locomo-results-{run_label}.json')
answer_results = Path(f'/content/locomo/results/locomo-answer-results-{run_label}.json')
report = Path(f'/content/locomo/results/locomo-comparison-{run_label}.md')
if summary.exists():
    print('ingest summary')
    print(json.dumps(json.loads(summary.read_text()), indent=2)[:4000])
if results.exists():
    print('retrieval evaluation summary')
    print(json.dumps(json.loads(results.read_text())['summary'], indent=2))
if answer_results.exists():
    print('answer evaluation summary')
    print(json.dumps(json.loads(answer_results.read_text())['summary'], indent=2))
if report.exists():
    print('comparison report')
    print(report.read_text()[:4000])

This notebook benchmarks the uploaded DB at `hf://<HF_REPO_ID>/data/locomo-psm-memory-ingest.db`. It defaults to a 20-question strict-JSON answer-evaluation smoke test to control token spend. Review `locomo-answer-results-smoke-20-json-v2.json`; set `ANSWER_LIMIT = 0` only when the smoke accuracy, context items, and judge reasoning look acceptable.